# Pet Gait Screening — Local Working Notebook

Open this in VS Code (Python + Jupyter extensions installed). Run cells top to bottom.

Before running: make sure your virtual environment is set up and selected as this notebook's kernel —
in VS Code, click the kernel picker in the top-right of this notebook and choose the `venv` you created with
`python -m venv venv` + `pip install -r requirements.txt`.

## 0. Sanity check — synthetic data (no video needed)
Confirms the gait-metric math and classifier logic work in this environment before touching real footage.

In [ ]:
import sys
sys.path.append('../src')

%run ../src/test_synthetic.py

## 1. Check the dataset
Runs the dataset checker: every file on disk must have a row in its label CSV, and label values must be allowed ones. Fix anything it reports before processing clips.

In [ ]:
import pandas as pd
from pathlib import Path

!python ../src/check_dataset.py --data ../data

gait_labels = pd.read_csv('../data/gait/gait_labels.csv')
display(gait_labels)

## 2. Run the pipeline on ONE real clip first
Don't batch-process everything blind — verify one clip works end-to-end, check the skeleton overlay looks right,
before running the rest.

In [ ]:
from pipeline import run

test_video = '../data/gait/videos/healthy/PUT_ONE_FILENAME_HERE.mp4'  # <-- edit this
report, result = run(test_video, out_dir='../outputs')

## 3. View the result inline
Check the gait chart and the flag/confidence/suspected leg right here before moving on.

In [ ]:
from IPython.display import Image, display
import glob

chart = sorted(glob.glob('../outputs/*_gait_report.png'))[-1]
display(Image(filename=chart))

print(f"Flag: {result.flag} | Confidence: {result.confidence} | Suspected leg: {result.suspected_leg}")
for r in result.reasons:
    print(" -", r)

## 4. Once step 2 looks correct — batch process every gait clip
Only run this after confirming the single-clip test above worked and the skeleton overlay tracked correctly.

In [ ]:
import glob

all_results = {}
for folder in ['healthy', 'limping']:
    for video_path in glob.glob(f'../data/gait/videos/{folder}/*.mp4'):
        print(f"\n{'='*60}\nProcessing: {video_path}\n{'='*60}")
        r, res = run(video_path, out_dir='../outputs')
        all_results[video_path] = (r, res)

## 5. Summary table across all processed clips
Quick overview once you've run a batch — useful for spotting which clips need a second look.

In [ ]:
rows = []
for path, (r, res) in all_results.items():
    rows.append({
        'video': Path(path).name,
        'flag': res.flag,
        'confidence': res.confidence,
        'suspected_leg': res.suspected_leg,
        'warnings': '; '.join(r.warnings) if r.warnings else ''
    })

summary_df = pd.DataFrame(rows)
display(summary_df)